<a href="https://colab.research.google.com/github/ydchen17/PPI_Analysis_Project/blob/main/ID_conversion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:human
! wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:yeast
#! wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:fruit%20fly
#! wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:mouse

--2021-10-13 00:35:29--  http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:human
Resolving www.ebi.ac.uk (www.ebi.ac.uk)... 193.62.193.80
Connecting to www.ebi.ac.uk (www.ebi.ac.uk)|193.62.193.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘species:human’

species:human           [              <=>   ]  64.46M  91.2KB/s    in 12m 45s 

2021-10-13 00:48:17 (86.3 KB/s) - ‘species:human’ saved [67592187]

--2021-10-13 00:48:17--  http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:yeast
Resolving www.ebi.ac.uk (www.ebi.ac.uk)... 193.62.193.80
Connecting to www.ebi.ac.uk (www.ebi.ac.uk)|193.62.193.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘species:yeast’

species:yeast           [       <=>          ]  38.44M  62.1KB/s    in 4m 15s  

2021-10-13 00:52:35 (155 KB/s) - ‘spec

In [ ]:
# just use the function below.

In [ ]:
import urllib.parse
import urllib.request

# basic ID conversion function
def ID_conversion(gene_list, from_type, to_type):
  gene_list_tab = ""
  for i in gene_list:
    gene_list_tab += str(i)+" "  
  params = {
      'from': from_type,
      'to': to_type,
      'format': 'tab',
      'query': gene_list_tab
      }
  data = urllib.parse.urlencode(params)
  data = data.encode('utf-8')
  req = urllib.request.Request('https://www.uniprot.org/uploadlists/', data)
  with urllib.request.urlopen(req) as f:
        response = f.read()
  df = pd.DataFrame([x.split('\t') for x in response.decode('utf-8').split('\n')])
  df.columns = df.iloc[0]
  return df[1:][:-1]

In [ ]:
# ID_conversion(genelist, 'ID', "P_ENTREZGENEID")
# ID_conversion(genelist, 'ID', "ENSEMBLGENOME_ID")
# ID_conversion(["dsadas"], 'ID', "P_ENTREZGENEID") # will return no result

# ID_conversion(interactorA, 'ID', "P_ENTREZGENEID").to_csv("You location here")
# ID_conversion(interactorA, 'ID', "P_ENTREZGENEID")[["From"]].to_csv("You location here")
# ID_conversion(interactorA, 'ID', "P_ENTREZGENEID")[["To"]].to_csv("You location here")

In [ ]:
import pandas as pd
human_ppi_edges = pd.read_csv("/content/species:human", sep = "\t", header = None)[[0,1]]
yeast_ppi_edges = pd.read_csv("/content/species:yeast", sep = "\t", header = None)[[0,1]]
#mouse_ppi_edges = pd.read_csv("/content/species:mouse", sep = "\t", header = None)[[0,1]]
#fly_ppi_edges = pd.read_csv("/content/species:fruit fly", sep = "\t", header = None)[[0,1]]

In [ ]:
# Edge list to node list
def nodelist(edgelist):
  nodelist = list(set(edgelist[0].values.tolist()+(edgelist[1].values.tolist())))
  nodelist = pd.DataFrame(nodelist)[0].str.split(":", expand=True)
  nodelist = nodelist.replace("uniprotkb", "ACC+ID")
  nodelist = nodelist.replace("refseq", "P_REFSEQ_AC")
  nodelist = nodelist.replace("ensembl", "ENSEMBL_ID")
  nodelist = nodelist.replace("chebi", "CHEMBL_ID")
  return nodelist

# nodelist(yeast_ppi_edges)

In [ ]:
def node_ID_convert(nodelist, to_type):
  result_list =[]
  for group_label, group_data in nodelist.groupby(0):
    print(group_label)
    result_intrim = ID_conversion(list(group_data[1]), group_label, "ID").rename(columns={"From":"Original_ID", "To":"Interim_ID"})
    result_final = ID_conversion(result_intrim["Interim_ID"], "ID", to_type).rename(columns={"From":"Interim_ID", "To":"Target_ID"})
    result = result_intrim.join(result_final.set_index("Interim_ID"), on="Interim_ID")
    result_list.append(result)
    print(result)
  result_df = pd.concat(result_list)
  return result_df.reset_index(drop=True)

In [ ]:
yeast_ppi_converted = node_ID_convert(nodelist(yeast_ppi_edges), "P_ENTREZGENEID")

ACC+ID
0    Original_ID   Interim_ID Target_ID
1         Q04629   SWF1_YEAST    851704
2         Q04949   DYN3_YEAST    855345
3         P23287  PP2B1_YEAST    851153
4         P46367  ALDH4_YEAST    854556
5         P42844  ZIM17_YEAST    855406
...          ...          ...       ...
4989      P24088  YRF18_YEAST    851189
4989      P24088  YRF18_YEAST    852158
4989      P24088  YRF18_YEAST    854577
4990      P32469   DPH5_YEAST    850869
4991      Q12446  LAS17_YEAST    854353

[5178 rows x 3 columns]
P_REFSEQ_AC
0 Original_ID   Interim_ID Target_ID
1   NP_010706  RL12A_YEAST    852026
1   NP_010706  RL12A_YEAST    856656
2   NP_010706  RL12B_YEAST    852026
2   NP_010706  RL12B_YEAST    856656
intact
Empty DataFrame
Columns: [Original_ID, Interim_ID, Target_ID]
Index: []


In [ ]:
yeast_ppi_converted

,Original_ID,Interim_ID,Target_ID
0,Q02805,ROD1_YEAST,854183
1,Q12078,SMF3_YEAST,850721
2,P30775,RF1M_YEAST,852734
3,P17536,TPM1_YEAST,855645
4,P36024,SIS2_YEAST,853946
...,...,...,...
5177,Q99219,YI31A_YEAST,NaN
5178,NP_010706,RL12A_YEAST,852026
5179,NP_010706,RL12A_YEAST,856656
5180,NP_010706,RL12B_YEAST,852026


In [ ]:
yeast_node_

In [ ]:
! pip install gprofiler-official
# get ortholog list
def get_orth(nodelist):
  from gprofiler import GProfiler
  gp = GProfiler(return_dataframe=True)
  orth = gp.orth(organism='scerevisiae',
            query=nodelist,
            target='hsapiens')
  return orth

In [ ]:
yeast_to_human = get_orth(nodelist(yeast_ppi_edges)[1].values.tolist())[["incoming","converted","ortholog_ensg","name"]]
yeast_to_human_converted = ID_conversion(yeast_to_human['ortholog_ensg'].values.tolist(), "ENSEMBL_ID", "SWISSPROT")
yeast_to_human_converted = yeast_to_human_converted.rename(columns={"To":"homolog_uniprot", "From":"ortholog_ensg"})
yeast_to_human = yeast_to_human.join(yeast_to_human_converted.set_index("ortholog_ensg"), on = "ortholog_ensg")

In [ ]:
yeast_to_human_converted

,ortholog_ensg,homolog_uniprot
1,ENSG00000006534,P43353
2,ENSG00000072210,P51648
3,ENSG00000108602,P30838
4,ENSG00000111275,P05091
5,ENSG00000118514,Q9H2A2
...,...,...
3293,ENSG00000127837,Q13685
3294,ENSG00000174145,Q9ULI1
3295,ENSG00000117543,Q9H2P9
3296,ENSG00000122574,A6NGB9


In [ ]:
yeast_to_human.to_csv("/content/drive/MyDrive/Network_GroupProject/conservation_lists/evo_table.csv")

In [ ]:
yeast_to_human

,incoming,converted,ortholog_ensg,name,homolog_uniprot
0,Q04629,YDR126W,N/A,N/A,NaN
1,Q04949,YMR299C,N/A,N/A,NaN
2,P23287,YLR433C,N/A,N/A,NaN
3,P46367,YOR374W,ENSG00000006534,ALDH3B1,P43353
4,P46367,YOR374W,ENSG00000072210,ALDH3A2,P51648
...,...,...,...,...,...
8246,P24088,nan,N/A,N/A,NaN
8247,P32469,YLR172C,ENSG00000117543,DPH5,Q9H2P9
8248,Q12446,YOR181W,ENSG00000115935,WIPF1,O43516
8249,Q12446,YOR181W,ENSG00000122574,WIPF3,A6NGB9


In [ ]:
yeast_conserved = set(yeast_to_human.dropna()["incoming"].values.tolist())

In [ ]:
len(yeast_conserved) 

2164

In [ ]:
with open("/content/drive/MyDrive/Network_GroupProject/conservation_lists/yeast_conserved.txt", "w") as f:
  for i in yeast_conserved:
    f.write(i+"\n")

In [ ]:
yeast_all = set(nodelist(yeast_ppi_edges)[1].values.tolist())

In [ ]:
len(yeast_all)

4972

In [ ]:
with open("/content/drive/MyDrive/Network_GroupProject/conservation_lists/yeast_all.txt", "w") as f:
  for i in yeast_all:
    f.write(i+"\n")

In [ ]:
yeast_specific = [node for node in yeast_all if node not in yeast_conserved]

In [ ]:
len(yeast_specific)

2808

In [ ]:
with open("/content/drive/MyDrive/Network_GroupProject/conservation_lists/yeast_specific.txt", "w") as f:
  for i in yeast_specific:
    f.write(i+"\n")

In [ ]:
human_conserved_all = set(yeast_to_human["homolog_uniprot"].values.tolist())
len(human_conserved_all)

3291

In [ ]:
human_all = set(nodelist(human_ppi_edges)[1].values.tolist())
len(human_all)

13359

In [ ]:
with open("/content/drive/MyDrive/Network_GroupProject/conservation_lists/human_all.txt", "w") as f:
  for i in human_all:
    f.write(str(i)+"\n")

In [ ]:
human_conserved = [node for node in human_conserved_all if node in human_all]
len(human_conserved)

2214

In [ ]:
with open("/content/drive/MyDrive/Network_GroupProject/conservation_lists/human_conserved.txt", "w") as f:
  for i in human_conserved:
    f.write(str(i)+"\n")

In [ ]:
human_specific = [node for node in human_all if node not in human_conserved_all]
len(human_specific)

11145

In [ ]:
with open("/content/drive/MyDrive/Network_GroupProject/conservation_lists/human_specific.txt", "w") as f:
  for i in human_specific:
    f.write(str(i)+"\n")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%bash
cd /content/drive/MyDrive/Network_GroupProject/conservation_lists/
wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:human 
wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:yeast
wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:fruit%20fly
wget http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:mouse

--2021-10-13 01:12:50--  http://www.ebi.ac.uk/Tools/webservices/psicquic/mint/webservices/current/search/query/species:human
Resolving www.ebi.ac.uk (www.ebi.ac.uk)... 193.62.193.80
Connecting to www.ebi.ac.uk (www.ebi.ac.uk)|193.62.193.80|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘species:human’

     0K .......... .......... .......... .......... ..........  198K
    50K .......... .......... .......... .......... ..........  517K
   100K .......... .......... .......... .......... ..........  569K
   150K .......... .......... .......... .......... .......... 35.1M
   200K .......... .......... .......... .......... .......... 2.63M
   250K .......... .......... .......... .......... ..........  682K
   300K .......... .......... .......... .......... .......... 2.73M
   350K .......... .......... .......... .......... .......... 21.9K
   400K .......... .......... .......... .......... ..........  273K
   450K .....

In [ ]:
human_nodelist = list(set(human_ppi_edges[0].values.tolist()+(human_ppi_edges[1].values.tolist())))
human_nodelist[:10]

['uniprotkb:P41212',
 'uniprotkb:Q9H9J4-2',
 'uniprotkb:Q62083',
 'uniprotkb:Q61881',
 'uniprotkb:Q96GY3',
 'uniprotkb:Q13616',
 'uniprotkb:Q8WVX3',
 'uniprotkb:P49768',
 'uniprotkb:Q6PEY2',
 'uniprotkb:Q96PD7']

In [ ]:
yeast_nodelist = list(set(yeast_ppi_edges[0].values.tolist()+(yeast_ppi_edges[1].values.tolist())))
yeast_nodelist[:10]

['uniprotkb:Q04629',
 'uniprotkb:Q04949',
 'uniprotkb:P23287',
 'uniprotkb:P46367',
 'uniprotkb:P42844',
 'uniprotkb:P39552',
 'uniprotkb:Q06497',
 'uniprotkb:Q06389',
 'uniprotkb:P32357',
 'uniprotkb:Q06631']

In [ ]:
human_ppi_edges = pd.read_csv("/content/species:human", sep = "\t", header = None)[[0,1]]
human_ppi_edges[0] = human_ppi_edges[0].str.split(":", expand=True)[1]
human_ppi_edges[1] = human_ppi_edges[1].str.split(":", expand=True)[1]
human_ppi_edges.to_csv("/content/drive/MyDrive/Network_GroupProject/conservation_lists/human_ppi_edges.csv", header=False, index=False)

In [ ]:
yeast_ppi_edges = pd.read_csv("/content/species:yeast", sep = "\t", header = None)[[0,1]]
yeast_ppi_edges[0] = yeast_ppi_edges[0].str.split(":", expand=True)[1]
yeast_ppi_edges[1] = yeast_ppi_edges[1].str.split(":", expand=True)[1]
yeast_ppi_edges.to_csv("/content/drive/MyDrive/Network_GroupProject/conservation_lists/yeast_ppi_edges.csv", header=False, index=False)

In [ ]:
library(plotly) 
p <- dotplot(human_specific_ego, showCategory=30) 
ggplotly(p)

In [ ]:
human_all_entrez = ID_conversion(human_all, 'ACC+ID', "P_ENTREZGENEID") 
human_conserved_entrez = ID_conversion(human_conserved, 'ACC+ID', "P_ENTREZGENEID") 
human_specific_entrez = ID_conversion(human_specific, 'ACC+ID', "P_ENTREZGENEID") 
yeast_all_entrez = ID_conversion(yeast_all, 'ACC+ID', "P_ENTREZGENEID") 
yeast_conserved_entrez = ID_conversion(yeast_conserved, 'ACC+ID', "P_ENTREZGENEID") 
yeast_specific_entrez = ID_conversion(yeast_specific, 'ACC+ID', "P_ENTREZGENEID") 

In [ ]:
def save_list(ID_list, filename):
  with open(filename, "w") as f:
    for i in ID_list:
      f.write(i+"\n")

In [ ]:
save_list(human_all_entrez["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/entrez/human_all_entrez.txt")
save_list(human_conserved_entrez["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/entrez/human_conserved_entrez.txt")
save_list(human_specific_entrez["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/entrez/human_specific_entrez.txt")
save_list(yeast_all_entrez["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/entrez/yeast_all_entrez.txt")
save_list(yeast_conserved_entrez["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/entrez/yeast_conserved_entrez.txt")
save_list(yeast_specific_entrez["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/entrez/yeast_specific_entrez.txt")

In [ ]:
human_all_GENENAME = ID_conversion(human_all, 'ACC+ID', "GENENAME") 
human_conserved_GENENAME = ID_conversion(human_conserved, 'ACC+ID', "GENENAME") 
human_specific_GENENAME = ID_conversion(human_specific, 'ACC+ID', "GENENAME") 
yeast_all_GENENAME = ID_conversion(yeast_all, 'ACC+ID', "GENENAME") 
yeast_conserved_GENENAME = ID_conversion(yeast_conserved, 'ACC+ID', "GENENAME") 
yeast_specific_GENENAME = ID_conversion(yeast_specific, 'ACC+ID', "GENENAME") 

In [ ]:
yeast_specific_GENENAME

,From,To
1,Q12345,IES3
2,P49367,LYS4
3,P53933,APP1
4,P40342,SWM2
5,Q03855,TY1B-DR1
...,...,...
2869,Q12143,NSL1
2870,Q06508,LOA1
2871,P38723,COS8
2872,P53156,YGL081W


In [ ]:
save_list(human_all_GENENAME["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/gene_name/human_all_GENENAME.txt")
save_list(human_conserved_GENENAME["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/gene_name/human_conserved_GENENAME.txt")
save_list(human_specific_GENENAME["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/gene_name/human_specific_GENENAME.txt")
save_list(yeast_all_GENENAME["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/gene_name/yeast_all_GENENAME.txt")
save_list(yeast_conserved_GENENAME["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/gene_name/yeast_conserved_GENENAME.txt")
save_list(yeast_specific_GENENAME["To"], "/content/drive/MyDrive/Network_GroupProject/conservation_lists/gene_name/yeast_specific_GENENAME.txt")

In [ ]:
import pandas as pd
human_expression = pd.read_csv("/content/drive/MyDrive/Network_GroupProject/conservation_lists/expression_avg/human_expression.txt",skiprows=11, sep="\t")[["string_external_id","abundance"]]
human_expression

,string_external_id,abundance
0,9606.ENSP00000359240,1.190
1,9606.ENSP00000359596,0.556
2,9606.ENSP00000358965,0.002
3,9606.ENSP00000303057,8.250
4,9606.ENSP00000222157,0.203
...,...,...
19944,9606.ENSP00000258821,1.470
19945,9606.ENSP00000355961,4.330
19946,9606.ENSP00000368190,3.500
19947,9606.ENSP00000360147,1.310


In [ ]:
yeast_expression = pd.read_csv("/content/drive/MyDrive/Network_GroupProject/conservation_lists/expression_avg/yeast_expression.txt",skiprows=11, sep="\t")[["string_external_id","abundance"]]
yeast_expression

,string_external_id,abundance
0,4932.YLR201C,34.20
1,4932.YMR188C,49.90
2,4932.YLL002W,11.70
3,4932.YNL015W,127.00
4,4932.YHR022C,1.47
...,...,...
6435,4932.YOR034C,7.65
6436,4932.YNL024C,13.80
6437,4932.YKL170W,37.20
6438,4932.YDL237W,40.70


In [ ]:
# yeast_expression["string_external_id"] = yeast_expression["string_external_id"].str.split(".", expand=True)[1]

In [ ]:
Interim_ID_table = ID_conversion(yeast_expression["string_external_id"].values.tolist(), "STRING_ID", "ID").rename(columns={"From":"string_external_id", "To":"Interim_ID"})
Final_ID_table = ID_conversion(Interim_ID_table["Interim_ID"].values.tolist(), "ID", "ACC").rename(columns={"From":"Interim_ID", "To":"UNIPROT"})
final_df = Interim_ID_table.set_index("Interim_ID").join(Final_ID_table.set_index("Interim_ID")).set_index("string_external_id").join(yeast_expression.set_index("string_external_id"))
final_df.to_csv("/content/drive/MyDrive/Network_GroupProject/conservation_lists/expression_avg/yeast_expression_converted.txt")

In [ ]:
Interim_ID_table = ID_conversion(human_expression["string_external_id"].values.tolist(), "STRING_ID", "ID").rename(columns={"From":"string_external_id", "To":"Interim_ID"})
Final_ID_table = ID_conversion(Interim_ID_table["Interim_ID"].values.tolist(), "ID", "ACC").rename(columns={"From":"Interim_ID", "To":"UNIPROT"})
final_df = Interim_ID_table.set_index("Interim_ID").join(Final_ID_table.set_index("Interim_ID")).set_index("string_external_id").join(human_expression.set_index("string_external_id"))
final_df.to_csv("/content/drive/MyDrive/Network_GroupProject/conservation_lists/expression_avg/human_expression_converted.txt")